In [0]:
from pyspark.sql.functions import sum, count, avg, to_date

# Step 1: Read cleaned data from Silver table
silver_table = "workspace.ecommerce.silver_events"
silver_df = spark.read.table(silver_table)

# Step 2: Business aggregation example
# Aggregate daily sales and event counts by product and event type
gold_df = (
    silver_df
    .withColumn("event_date", to_date("event_time"))
    .groupBy("event_date", "product_id", "event_type")
    .agg(
        sum("price").alias("total_sales"),
        count("event_type").alias("event_count"),
        avg("price").alias("avg_price")
    )
)

# Step 3: Write to Gold Delta table (overwrite for initial run)
gold_table = "workspace.ecommerce.gold_events"
gold_df.write.format("delta").mode("overwrite").saveAsTable(gold_table)

# Step 4: Display sample Gold data
display(spark.read.table(gold_table).limit(10))